> 📄 English: [building-chatbot-notebook.ipynb](building-chatbot-notebook.ipynb)

# 用 DeepSeek 和 Streamlit 构建聊天机器人 Agent

> 本文是官方教程 [building-chatbot-notebook.ipynb](building-chatbot-notebook.ipynb) 的中文翻译版，并把后端从 OpenAI 改成了 DeepSeek（OpenAI 兼容接口），方便使用 DeepSeek key 直接运行。

## 简介

想不想自己动手做一个像 ChatGPT 一样的聊天机器人？本教程将使用 DeepSeek 的 API（OpenAI 兼容接口，提供强大的语言模型）和 Streamlit（一个用 Python 写 Web 应用的库），从零开始搭建一个简单的 AI 聊天机器人。

学完本教程，你将得到一个能在浏览器里运行的入门级聊天机器人：带聊天界面，能对话，还能上传文件给机器人分析。我们会一步步讲解每一段代码，即使你是零基础也能看懂。

**我们要构建的东西：**
- 一个 Web 应用，你可以在里面给 AI 发消息并收到回复
- 界面类似聊天窗口（就像微信这样的聊天软件）
- 一个上传文件的选项（比如上传文本文件），让聊天机器人使用

本教程的重点是：用 Streamlit 做干净简洁的界面，并把 DeepSeek 的 API 接进来，负责机器人的「智能」部分。开始吧！

## 环境要求

开始之前，请确保你有：

- 系统里装了 Python 3.x
- 一个 DeepSeek API key（访问 DeepSeek 语言模型所必需）。到 DeepSeek 开放平台注册账号，创建 API key 即可；如果你走的是公司内网网关，请用网关地址和对应的 key
- 基础的 Python 知识（函数、变量）——我们会解释所有内容，但懂一点 Python 语法会更有帮助
- 用到的库：`openai`（通过 OpenAI 兼容接口访问 DeepSeek）和 `streamlit`（搭建 Web 界面）

## 安装所需库

用 pip 安装。在终端/命令行里运行：

In [ ]:
!pip install openai streamlit

如果你还需要处理文件，可以额外安装：

In [ ]:
!pip install streamlit openai requests PyPDF2

## 配置：API Key

要调用 DeepSeek 的接口，你需要提供 API key 用于鉴权。有两种方式：

1. **方式一（推荐）**：把 API key 存成环境变量（例如 `DEEPSEEK_API_KEY`）。这样 key 就不会写死在代码里。
   - Linux/Mac：在终端执行 `export DEEPSEEK_API_KEY='你的key'`
   - Windows：在命令行执行 `set DEEPSEEK_API_KEY="你的key"`

2. **方式二**：直接在代码里写死 API key（测试方便，但分享代码时容易泄露，不推荐）

本教程默认你把 key 存成了环境变量，这是最佳实践，避免硬编码密钥。

为了支持方式一，需要安装 python-dotenv 包：

In [ ]:
!pip install python-dotenv

## 构建 AI Agent（接入 DeepSeek）

先写一小段 Python 与 DeepSeek 接口交互。这就是聊天机器人的「大脑」——把用户消息发给 DeepSeek，拿回回复。

In [ ]:
from dotenv import load_dotenv
import openai
import os

load_dotenv()  # 从 .env 加载环境变量

# 用 OpenAI 兼容接口访问 DeepSeek
client = openai.OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://ai-api.d-robotics.cc/v1",  # 内网网关地址（公网 DeepSeek 用 https://api.deepseek.com）
)

def generate_response(user_prompt):
    """
    把用户输入发给 DeepSeek，返回 AI 的回复。

    参数
    ----
    user_prompt : str
        用户输入的消息

    返回
    ----
    str
        AI 生成的回复（纯文本）
    """
    response = client.chat.completions.create(
        model="deepseek-flash",  # 使用的模型（内网网关提供的 DeepSeek 模型）
        messages=[{"role": "user", "content": user_prompt}],
    )
    message_text = response.choices[0].message.content
    return message_text

# 快速测试：问一句看函数是否正常工作
test_reply = generate_response("你好，今天过得怎么样？")
print(test_reply)

拆解一下 `generate_response` 里发生了什么：

- 我们调用 `client.chat.completions.create(...)`，传入模型名和一个消息列表。`messages` 参数期望一个对话历史。这里只给了一条消息——用户的输入——并把它的角色标为 `"user"`。你也可以加一条 `"system"` 消息来设定 AI 的行为（比如告诉它扮演一个友好的助手），但这里先保持简单。
- 我们选了模型 `"deepseek-flash"`。如果你用的是公网 DeepSeek，可以把模型换成 `"deepseek-chat"`。
- 接口返回一个 response 对象，里面包含 AI 的回复。回复文本藏在 `response.choices[0].message.content` 里，我们取出来返回。
- 最后的 `print(test_reply)` 只是用来确认一切正常，会在控制台打印 AI 的回答。在 Streamlit 应用里我们不用 print，这只是个冒烟测试。

到这里，如果你在一个普通 Python 环境里运行这段代码（确保 key 设置好了），就能看到 AI 返回的一段文字。这证明 DeepSeek 接口已打通。AI 部分就绪后，接下来用 Streamlit 搭 Web 界面。

## 构建 Streamlit UI

Streamlit 让你只写 Python 就能做出交互式 Web 界面——不需要写 HTML 或 JavaScript。我们要做一个聊天界面：

- 页面有标题、布局干净
- 侧边栏有文件上传组件（用户可以上传文本或 PDF）
- 主区域显示对话（用户和助手消息）
- 底部有输入框让用户输入新消息

一步步来。

### 1. 设置 Streamlit 页面布局

先初始化 Streamlit 应用配置，并在页面顶部加上标题。

In [ ]:
import streamlit as st  # 引入 Streamlit 做 UI

# 设置页面配置
st.set_page_config(
    page_title="AI Chatbot",   # 浏览器标签页标题
    page_icon="🤖",           # 页面图标（机器人 emoji）
    layout="wide"             # 使用全宽布局
)

# 添加标题
st.title("🤖 AI Chatbot Assistant")
st.markdown("**欢迎！** 随便问点什么，或者上传文件让机器人分析。")

说明：
- `st.set_page_config(...)` 设置应用的全局配置。我们给应用设了浏览器标签页标题、一个 emoji 图标，并把布局设为 `"wide"`，让聊天区域能用更多横向空间。
- `st.title("...")` 在页面顶部显示一个大标题，标题里也加了 emoji 作为装饰。
- `st.markdown("**欢迎！** ...")` 用 Markdown 加粗显示一句欢迎语。

### 2. 添加文件上传组件

接下来加一个文件上传组件，让用户可以上传文件（比如文本文件）给聊天机器人用。我们把上传组件放在侧边栏，保持主界面干净。

In [ ]:
# 侧边栏的文件上传区域
uploaded_file = st.sidebar.file_uploader(
    "Upload a file (optional):",  # 上传组件的标签
    type=["txt", "pdf"]           # 只允许文本和 PDF 两种类型
)

# 如果上传了文件，读取或处理它（这里只显示文件名作为确认）
if uploaded_file is not None:
    file_details = f"**{uploaded_file.name}** ({uploaded_file.size} bytes)"
    st.sidebar.write("Uploaded file:", file_details)
    # 在真实应用里，你可以把文件内容传给 AI 或做其他处理。
    # 例如文本文件可以这样读：text = uploaded_file.read().decode('utf-8')

说明：
- `st.sidebar.file_uploader(...)` 在侧边栏创建文件上传组件。`type` 参数限制了允许的文件后缀（这里只允许 .txt 和 .pdf）。
- 用 `if uploaded_file is not None:` 判断用户是否上传了文件。如果上传了，获取文件信息：
  - `uploaded_file.name` 是文件名
  - `uploaded_file.size` 是文件大小（字节）
- 用 `st.sidebar.write` 在侧边栏显示文件名和大小，文件名用 Markdown 加粗。

注意：Streamlit 的侧边栏可以被用户显示或隐藏。如果你运行后看不到上传组件，找一下页面上的小箭头或侧边栏开关，展开它即可。

### 3. 用会话状态保存聊天历史

聊天界面的一个重要点是「记住对话」。我们希望应用显示之前的消息和回复，而不只是最新一条。Streamlit 应用在每次用户交互时都会从头到尾重跑一遍脚本，所以如果不存状态，每条新消息都会丢掉之前的聊天历史。解决办法是用 Streamlit 的 session state 来存消息。

In [ ]:
# 如果还没有消息历史，就初始化它
if "messages" not in st.session_state:
    st.session_state.messages = []  # 用列表存所有消息（每个元素是带 'role' 和 'content' 的字典）

# （可选）让机器人开场先说一句问候语：
if not st.session_state.messages:
    st.session_state.messages.append({"role": "assistant", "content": "你好！有什么可以帮你的吗？随便问点什么，或者上传文件让我分析。"})

说明：
- 我们用 `st.session_state` 里的 `"messages"` 键来存对话。应用第一次运行时这个键不存在，所以初始化为空列表。
- `st.session_state.messages` 这个列表用来存消息，每条消息是一个字典，形如 `{"role": "user", "content": "你好"}` 或 `{"role": "assistant", "content": "你好，有什么可以帮你？"}`。对话增长时我们就往这个列表里追加。
- 我们加了一个可选步骤：如果消息列表为空（说明还没有对话），就追加一条助手问候语。这样用户一打开应用就能看到机器人发来的欢迎消息。

这样使用 session state，应用就能记住之前的消息。每次脚本重跑（每次用户输入），它都会保留上一轮运行留下的 `st.session_state.messages`。

### 4. 显示对话

现在状态里有一个消息列表，我们要把它们以聊天样式显示在页面上。Streamlit 提供了 `st.chat_message`，正好用来显示用户或助手样式的聊天气泡。

In [ ]:
# 显示所有历史消息
for msg in st.session_state.messages:
    if msg["role"] == "assistant":
        # 助手消息靠左（默认样式）
        with st.chat_message("assistant"):
            st.markdown(msg["content"])
    else:
        # 用户消息靠右
        with st.chat_message("user"):
            st.markdown(msg["content"])

说明：
- 遍历消息列表里的每一条消息。
- 检查每条消息的 `"role"`。如果是 `"assistant"`，用 `st.chat_message("assistant")` 创建助手聊天气泡容器；如果是 `"user"`，用 `st.chat_message("user")`。
- `with st.chat_message(<角色>):` 打开一个该角色样式的聊天气泡容器，在这个 with 块里输出消息内容。我们用 `st.markdown` 渲染文本。
- Streamlit 会自动把「助手」消息和「用户」消息渲染成不同样式（比如不同背景色，用户消息可能靠右对齐）。

跑完这个循环后，应用就会按顺序渲染出所有历史消息，用户可以往上翻看对话历史。

### 5. 发送新消息（聊天输入框）

最后，需要一个输入框让用户输入新消息。Streamlit 的 `st.chat_input()` 会在页面底部提供一个固定位置的输入框，非常适合聊天应用。当用户输入消息并按回车后，我们会：

1. 捕获这条消息
2. 加到 session state 历史里
3. 发给 DeepSeek（用前面的 `generate_response` 函数）
4. 拿到 AI 回复并加入历史

然后应用会重跑，在上面的循环里显示更新后的对话（包括新消息）。

In [ ]:
# 聊天输入组件（出现在页面底部）
user_message = st.chat_input("Type your message here...")

if user_message:
    # 1. 把用户消息加入聊天历史
    st.session_state.messages.append({"role": "user", "content": user_message})

    # 2. 立即显示用户消息（让它马上出现在上方聊天区）
    with st.chat_message("user"):
        st.markdown(user_message)

    # 3. 生成 AI 回复（这可能需要一点时间）
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):  # 等待 AI 时显示加载动画
            assistant_reply = generate_response(user_message)
            st.markdown(assistant_reply)
    # 4. 把助手回复加入聊天历史
    st.session_state.messages.append({"role": "assistant", "content": assistant_reply})

说明：
- `user_message = st.chat_input("...")` 在底部创建输入框。用户提交消息时，`st.chat_input` 把消息作为字符串返回。
- `if user_message:` 当用户刚发了消息时为真。
- 然后按步骤：
  1. 把用户消息以 `"user"` 角色追加到历史列表。
  2. 立即显示用户消息，让用户马上在聊天区看到自己的消息。
  3. 用 `st.chat_message("assistant")` 上下文放即将到来的回复，在里面用 `st.spinner("Thinking...")` 在调用 API 时显示加载动画，调用 `generate_response(user_message)` 拿 AI 回复并显示。
  4. 把助手回复以 `"assistant"` 角色追加到历史列表。

## 完整代码

下面是把所有片段合并后的完整脚本：

In [ ]:
# app.py（合并所有步骤的完整代码）

import os
import openai
import streamlit as st

# 用 OpenAI 兼容接口访问 DeepSeek
client = openai.OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://ai-api.d-robotics.cc/v1",  # 内网网关（公网 DeepSeek 用 https://api.deepseek.com）
)

def generate_response(user_prompt):
    response = client.chat.completions.create(
        model="deepseek-flash",
        messages=[{"role": "user", "content": user_prompt}],
    )
    message_text = response.choices[0].message.content
    return message_text

st.set_page_config(page_title="AI Chatbot", page_icon="🤖", layout="wide")
st.title("🤖 AI Chatbot Assistant")
st.markdown("**欢迎！** 随便问点什么，或者上传文件让机器人分析。")

uploaded_file = st.sidebar.file_uploader("Upload a file (optional):", type=["txt", "pdf"])
if uploaded_file is not None:
    st.sidebar.write("Uploaded file:", f"**{uploaded_file.name}** ({uploaded_file.size} bytes)")

if "messages" not in st.session_state:
    st.session_state.messages = []
if not st.session_state.messages:
    st.session_state.messages.append({"role": "assistant", "content": "你好！有什么可以帮你的吗？"})

# 显示历史消息
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# 聊天输入组件
if user_msg := st.chat_input("Type your message here..."):
    st.session_state.messages.append({"role": "user", "content": user_msg})
    with st.chat_message("user"):
        st.markdown(user_msg)
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            assistant_msg = generate_response(user_msg)
            st.markdown(assistant_msg)
    st.session_state.messages.append({"role": "assistant", "content": assistant_msg})

## 运行应用

现在代码写好了（例如存成 app.py 文件），可以运行 Streamlit 应用了。在终端里，确保当前目录包含 app.py，然后运行：

In [ ]:
# 在终端里运行，不要在这个 notebook 里运行
# streamlit run app.py

这条命令会启动 Streamlit 开发服务器，并在浏览器里打开应用（通常是 http://localhost:8501）。你会看到带标题的聊天机器人界面，以及助手发来的开场问候。

试试看：在聊天框里输入一个问题（比如「法国的首都是哪里？」）然后按回车。你会看到你的消息出现在右边，稍等片刻后 AI 的回复会出现在左边（等待回复期间应用可能显示「Thinking...」）。

注意：运行应用时要确保你的 DeepSeek key 有效、能访问接口，因为应用需要调用 DeepSeek 服务器来获取回复。

## Streamlit 聊天机器人界面

下面是聊天机器人运行后的界面效果：

![Streamlit Chatbot Interface](assets/streamlit_chatbot.png)

图中展示了带消息历史的聊天界面、底部的输入框，以及侧边栏的文件上传选项。

## Streamlit 聊天机器人演示

In [ ]:
from IPython.display import HTML

HTML("""
<video width="960" height="720" controls>
  <source src="assets/streamlit_chatbot_video.mp4" type="video/mp4">
  你的浏览器不支持 video 标签。
</video>
""")

这段视频展示了用户如何实时与聊天机器人交互、提问并收到回复。

## 结语

恭喜！你用 DeepSeek 和 Streamlit 做出了第一个 AI 聊天机器人！🎉 本教程我们覆盖了：

- 安装并配置所需库（streamlit 和 openai）
- 获取并使用 DeepSeek API key 访问语言模型
- 写一个与 DeepSeek 接口通信、获取回复的函数
- 用 Streamlit 搭建 Web 界面，包括聊天消息显示、输入框，以及侧边栏的文件上传组件
- 用 st.session_state 跨交互保存状态（对话历史），实现多轮对话
- 运行 Streamlit 应用并通过浏览器与聊天机器人交互

这个基础应用是一个很好的起点，你可以从很多方向扩展：

- **增强 AI 的上下文**：我们让 AI 调用保持无状态（只发送最新一条用户消息）。你可以把整段对话作为 messages 传给 chat.completions.create，让 AI 记住之前的提问。
- **使用上传的文件**：把文件内容接入对话。例如上传 PDF 时，提取文字并作为 system 消息前置。
- **UI 增强**：Streamlit 的聊天组件支持头像等，也可以加更多设计元素（颜色、侧边栏信息等）。
- **部署**：你可以轻松把这个应用部署到网上（例如用 Streamlit Cloud），把聊天机器人分享给朋友或同事。

用很少的代码，我们就做出了一个交互式 AI 助手！尽情实验、在此基础上扩展吧。祝你玩得开心！🚀